# 07. Numbers, Table 1 and Figure 1 for the SSAC abstract

Inputs: `pressure_events.csv.gz` (notebook 05) and the per-defender table rebuilt as in notebook 06.
- Table 1: agreement of every pressure definition with the analysts (P vs N): ROC-AUC with 95% CI,
  Cohen's κ at the threshold that flags the same share of passes as the analysts, presser top-1 accuracy
- Paired bootstrap: is the nearest-defender distance really better than the published metrics?
- Figure 1: share labelled 'pressured' by distance × defender speed (PFF speed), written thresholds marked

In [1]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import lightgbm as lgb
from sklearn.metrics import roc_auc_score, cohen_kappa_score

from football.common import KEY, cv_predict, boot_ci, boot_diff
from football.pressure import defender_table, pass_frames, PFF_R_PRESSURED, PFF_R_ATTEMPT, PFF_R_SPRINT

OUT = os.path.join("..", "outputs")
FIG = os.path.join(OUT, "figs")
E = pd.read_csv(os.path.join(OUT, "pressure_events.csv.gz"))
ev = pd.read_csv(os.path.join(OUT, "events.csv.gz"), low_memory=False)
print(f"passes: {len(E):,}  games: {E.game_id.nunique()}")
E.label.value_counts(normalize=True).round(3)

passes: 61,012  games: 64


label
N    0.500
P    0.366
L    0.076
A    0.058
Name: proportion, dtype: float64

## 1. Per-defender table (as in notebook 06)

In [2]:
pa = ev.query("event_type == 'PA' and setpiece == 'O' and pressure.notna()", engine="python")
f0 = pass_frames(OUT, pa[KEY])
frames = dict(tuple(f0.groupby(KEY)))
rows = []
for r in pa.itertuples():
    f = frames.get((r.game_id, r.pe_id))
    if f is None:
        continue
    c, D = f[f.player_id == r.actor_id], f[~f.atk]
    if c.empty or len(D) < 5:
        continue
    t = defender_table(c[["x", "y"]].to_numpy(float)[0], c[["vx", "vy"]].to_numpy(float)[0],
                       D[["x", "y"]].to_numpy(float), D[["vx", "vy"]].to_numpy(float),
                       f[f.atk & (f.player_id != r.actor_id) & (f.position != "GK")][["x", "y"]].to_numpy(float))
    rows.append(pd.DataFrame(dict(game_id=r.game_id, pe_id=r.pe_id, player_id=D.player_id.to_numpy(),
                                  label=r.pressure, presser=r.pressure_player_id, pff_speed=D.speed.to_numpy(), **t)))
DF = pd.concat(rows, ignore_index=True)
DF["behind"] = DF.cos_goal < 0
len(DF)

671067

## 2. Scores per pass for every definition
Binary rules: 4.57 m radius (common event-data rule); PFF written rule (2.74 / 4.57 m, closing > 0, radius ×0.6 from behind, 3.66 m at sprint).
Learned: LightGBM on carrier-centric tracking features, game-grouped CV (from notebook 05 features).

In [3]:
g = DF.groupby(KEY)
S = pd.DataFrame({
    "nearest distance": -g.dist.min(),
    "Andrienko et al. 2017": g.andr.sum(),
    "Pressing Intensity (Bekkers 2025)": 1 - np.exp(DF.assign(l=np.log1p(-DF.p_int.clip(upper=1 - 1e-12))).groupby(KEY).l.sum()),
    "min arrival time (Narizuka et al. 2026)": -g.arrival.min(),
})
S["4.57 m radius rule"] = (-S["nearest distance"] < PFF_R_ATTEMPT).astype(int)
scale = np.where(DF.behind, 0.6, 1.0)
d = DF.dist / scale
P = ((d < PFF_R_PRESSURED) & (DF.closing > 0)) | ((DF.dist < PFF_R_SPRINT) & (DF.closing > 4.0))
S["PFF written rule (P)"] = P.groupby([DF.game_id, DF.pe_id]).max().astype(int)

E2 = E.set_index(KEY)
m = E2.label.isin(["P", "N"])
y = (E2.label[m] == "P").astype(int)
CARRIER_FEATS = ["d1", "d2", "d3", "n3", "n5", "andr_sum", "andr_max", "pi_max", "pi_comb", "tti_min",
                 "arrival_min", "closing1", "cos1", "speed1", "closing_max", "carrier_speed", "cx", "cy"]
LGB = dict(n_estimators=300, learning_rate=0.05, num_leaves=15, min_child_samples=40,
           subsample=0.8, subsample_freq=1, colsample_bytree=0.8, reg_lambda=1.0, verbose=-1)
Xm = E2.loc[m, CARRIER_FEATS].fillna(-1).reset_index(drop=True)
learned = cv_predict(lambda s: lgb.LGBMClassifier(random_state=s, **LGB), Xm, y.reset_index(drop=True),
                     E2.loc[m].reset_index().game_id, seeds=(0,))
S = S.reindex(y.index)
S["learned from labels (LightGBM, CV)"] = learned
S.describe().T.round(3)

,count,mean,std,min,25%,50%,75%,max
nearest distance,52852.0,-5.589,4.593,-56.626,-7.617,-4.244,-2.226,-0.000
Andrienko et al. 2017,52852.0,0.386,0.378,0.000,0.016,0.303,0.653,3.106
Pressing Intensity (Bekkers 2025),52852.0,0.608,0.349,0.000,0.275,0.733,0.926,1.000
min arrival time (Narizuka et al. 2026),52852.0,-1.177,0.618,-6.299,-1.507,-1.037,-0.720,-0.056
4.57 m radius rule,52852.0,0.533,0.499,0.000,0.000,1.000,1.000,1.000
PFF written rule (P),52852.0,0.201,0.401,0.000,0.000,0.000,0.000,1.000
"learned from labels (LightGBM, CV)",52852.0,0.423,0.359,0.003,0.071,0.307,0.833,0.984


## 3. Table 1 — agreement with the analysts (P vs N)

In [4]:
def kappa_matched(y, s):
    if s.nunique() <= 2:
        return cohen_kappa_score(y, s)
    thr = np.quantile(s, 1 - y.mean())
    return cohen_kappa_score(y, s > thr)


pres = DF[(DF.label == "P") & DF.presser.notna()].copy()   # same population as the P vs N columns
pres["is_presser"] = pres.player_id == pres.presser
games = y.index.get_level_values("game_id")   # cluster bootstrap by game
pres = pres[pres.groupby(KEY).is_presser.transform("any")]
top1_col = {"nearest distance": -pres.dist, "Andrienko et al. 2017": pres.andr,
            "Pressing Intensity (Bekkers 2025)": pres.p_int, "min arrival time (Narizuka et al. 2026)": -pres.arrival}
top1 = {k: pres.loc[v.groupby([pres.game_id, pres.pe_id]).idxmax(), "is_presser"].mean()
        for k, v in top1_col.items()}

rows = []
for name in S.columns:
    s = S[name]
    binary = s.nunique() <= 2
    auc = np.nan if binary else roc_auc_score(y, s)
    ci = (np.nan, np.nan) if binary else boot_ci(y, s, roc_auc_score, n=500, groups=games)
    rows.append({"definition": name, "uses velocity": name.startswith(("Pressing", "min arrival", "PFF written", "learned")),
                 "ROC-AUC": auc, "95% CI": "" if binary else f"[{ci[0]:.3f}, {ci[1]:.3f}]",
                 "kappa": kappa_matched(y, s), "presser top-1": top1.get(name, np.nan)})
table1 = pd.DataFrame(rows).round(3)
table1

,definition,uses velocity,ROC-AUC,95% CI,kappa,presser top-1
0,nearest distance,False,0.888,"[0.876, 0.899]",0.655,0.778
1,Andrienko et al. 2017,False,0.864,"[0.852, 0.876]",0.617,0.750
2,Pressing Intensity (Bekkers 2025),True,0.795,"[0.784, 0.805]",0.446,0.640
3,min arrival time (Narizuka et al. 2026),True,0.874,"[0.862, 0.885]",0.612,0.757
4,4.57 m radius rule,False,NaN,,0.586,NaN
5,PFF written rule (P),True,NaN,,0.417,NaN
6,"learned from labels (LightGBM, CV)",True,0.912,"[0.902, 0.921]",0.674,NaN


## 4. Paired bootstrap: nearest distance vs each published metric

In [5]:
pairs = []
for other in ["Andrienko et al. 2017", "Pressing Intensity (Bekkers 2025)", "min arrival time (Narizuka et al. 2026)",
              "learned from labels (LightGBM, CV)"]:
    mdiff, ci, p_le0 = boot_diff(y, S["nearest distance"], S[other], roc_auc_score, n=500, groups=games)
    pairs.append(dict(comparison=f"distance − {other}", auc_diff=mdiff, ci95=f"[{ci[0]:.3f}, {ci[1]:.3f}]"))
pd.DataFrame(pairs).round(3)

,comparison,auc_diff,ci95
0,distance − Andrienko et al. 2017,0.024,"[0.022, 0.026]"
1,distance − Pressing Intensity (Bekkers 2025),0.093,"[0.086, 0.100]"
2,distance − min arrival time (Narizuka et al. 2...,0.014,"[0.012, 0.016]"
3,"distance − learned from labels (LightGBM, CV)",-0.024,"[-0.027, -0.021]"


## 4b. Sensitivity: Andrienko threat directions, Pressing Intensity aggregation, goalkeeper passes
- Andrienko et al. (2017) use the goal and attacking teammates as threat directions (max per presser). Main:
  goal + teammates closer to goal than the passer; variants: goal only, goal + all teammates.
- Bekkers (2025) Eq. 5 combines pressers as 1 − Π(1 − p) (main); variant: max over defenders.
- Narizuka et al. (2026) exclude goalkeeper possessions: AUC recomputed without goalkeeper passes.

In [6]:
var = {
    "Andrienko: goal only": g.andr_goal.sum(),
    "Andrienko: goal + teammates closer to goal (main)": g.andr.sum(),
    "Andrienko: goal + all teammates": g.andr_all.sum(),
    "Pressing Intensity: max over defenders": g.p_int.max(),
    "Pressing Intensity: Eq. 5 total (main)": S["Pressing Intensity (Bekkers 2025)"],
}
display(pd.Series({k: roc_auc_score(y, v.reindex(y.index)) for k, v in var.items()}, name="ROC-AUC P vs N").round(3))

pos = pd.read_csv(os.path.join(OUT, "players.csv.gz"), usecols=KEY + ["player_id", "position"])
gk = pa[KEY + ["actor_id"]].merge(pos, left_on=KEY + ["actor_id"], right_on=KEY + ["player_id"]).query("position == 'GK'")
keep = ~y.index.isin(gk.set_index(KEY).index)
print(f"goalkeeper passes in the P vs N set: {(~keep).sum():,}")
cols = [c for c in S.columns if S[c].nunique() > 2]
subsets = {"all passes": np.ones(len(y), bool),
           "excluding goalkeeper passes": keep,
           "excluding disallowed events (nonEvent)": ~E2.loc[y.index, "non_event"].to_numpy(),
           "only passes with trusted velocities": E2.loc[y.index, "vel_ok_pass"].to_numpy()}
print({k: int(v.sum()) for k, v in subsets.items()})
pd.DataFrame({k: [roc_auc_score(y[v], S[c][v]) for c in cols] for k, v in subsets.items()}, index=cols).round(3)

Andrienko: goal only                                 0.850
Andrienko: goal + teammates closer to goal (main)    0.864
Andrienko: goal + all teammates                      0.868
Pressing Intensity: max over defenders               0.790
Pressing Intensity: Eq. 5 total (main)               0.795
Name: ROC-AUC P vs N, dtype: float64

goalkeeper passes in the P vs N set: 2,652
{'all passes': 52852, 'excluding goalkeeper passes': 50200, 'excluding disallowed events (nonEvent)': 52362, 'only passes with trusted velocities': 41913}


,all passes,excluding goalkeeper passes,excluding disallowed events (nonEvent),only passes with trusted velocities
nearest distance,0.888,0.882,0.889,0.897
Andrienko et al. 2017,0.864,0.857,0.864,0.874
Pressing Intensity (Bekkers 2025),0.795,0.783,0.795,0.812
min arrival time (Narizuka et al. 2026),0.874,0.867,0.874,0.883
"learned from labels (LightGBM, CV)",0.912,0.907,0.912,0.920


## 5. Figure 1 — what the analysts actually use
Nearest defender only. Speed = PFF's stored speed (no direction).

In [7]:
near = DF.loc[DF.groupby(KEY).dist.idxmin()].set_index(KEY)
near = near[near.dist < 8]
near["dist_bin"] = pd.cut(near.dist, np.arange(0, 8.5, 1.0))
near["speed_bin"] = pd.cut(near.pff_speed, [-1, 1, 3, 20], labels=["< 1 m/s (static)", "1–3 m/s", "> 3 m/s"])
tab = near.pivot_table(index="dist_bin", columns="speed_bin", values="label",
                       aggfunc=lambda s: (s == "P").mean(), observed=True)
cnt = near.pivot_table(index="dist_bin", columns="speed_bin", values="label", aggfunc="size", observed=True)

fig, ax = plt.subplots(figsize=(5.2, 3.4))
x = [iv.mid for iv in tab.index]
for col, c in zip(tab.columns, ["#9ecae1", "#4292c6", "#08306b"]):
    ax.plot(x, tab[col], marker="o", ms=4, color=c, label=f"defender {col}")
for thr, txt in [(PFF_R_PRESSURED, "2.74 m"), (PFF_R_ATTEMPT, "4.57 m")]:
    ax.axvline(thr, ls=":", c="grey", lw=1)
    ax.text(thr + 0.08, 0.95, txt, fontsize=7, color="grey", va="top")
ax.set_xlabel("distance of nearest defender to passer (m)")
ax.set_ylabel("share labelled 'pressured' by analysts")
ax.set_ylim(0, 1); ax.legend(frameon=False, fontsize=7)
ax.set_title("Static defenders close by still count as pressure", fontsize=9)
fig.tight_layout(); fig.savefig(os.path.join(FIG, "abstract_fig1.png"), dpi=300)
display(tab.round(3))
cnt

speed_bin,< 1 m/s (static),1–3 m/s,> 3 m/s
dist_bin,,,
"(0.0, 1.0]",0.878,0.874,0.920
"(1.0, 2.0]",0.699,0.745,0.858
"(2.0, 3.0]",0.342,0.537,0.667
"(3.0, 4.0]",0.212,0.290,0.394
"(4.0, 5.0]",0.168,0.179,0.214
"(5.0, 6.0]",0.095,0.138,0.126
"(6.0, 7.0]",0.131,0.116,0.092
"(7.0, 8.0]",0.067,0.092,0.073


speed_bin,< 1 m/s (static),1–3 m/s,> 3 m/s
dist_bin,,,
"(0.0, 1.0]",98,365,703
"(1.0, 2.0]",146,1127,2526
"(2.0, 3.0]",187,1290,2595
"(3.0, 4.0]",189,1160,1983
"(4.0, 5.0]",149,936,1399
"(5.0, 6.0]",137,646,1004
"(6.0, 7.0]",130,586,703
"(7.0, 8.0]",105,510,519


## Save

In [8]:
table1.to_csv(os.path.join(OUT, "abstract_table1.csv"), index=False)
print("saved")

saved
